# 05 — Ансамблирование и итоги

Здесь мы берём предсказания всех трёх моделей и строим ансамбли:

| Метод | Описание |
|---|---|
| **Hard Voting** | Голосование большинством по классам |
| **Soft Voting** | Усреднение вероятностей |
| **Weighted Averaging** | Усреднение, взвешенное по F1-macro |
| **Stacking (LogReg)** | Метамодель — логистическая регрессия на вероятностях |
| **Stacking (SVM)** | Метамодель — SVM на вероятностях |

In [ ]:
import sys, os

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
print('Setup done')

## 1. Загрузка предсказаний всех трёх моделей

In [ ]:
MODEL_DIRS = {
    'ruBERT':      f'{WORKING_DIR}/models/rubert',
    'XLM-RoBERTa': f'{WORKING_DIR}/models/xlmroberta',
    'ruBERT-tiny': f'{WORKING_DIR}/models/rubert_tiny',
}

all_probs  = {}  # model_name -> (n_test, 3)
all_preds  = {}  # model_name -> (n_test,)
all_labels = None
all_val_f1 = {}

for model_name, model_dir in MODEL_DIRS.items():
    probs_path = os.path.join(model_dir, 'test_probs.npy')
    preds_path = os.path.join(model_dir, 'test_preds.npy')
    labels_path = os.path.join(model_dir, 'test_labels.npy')
    
    if not os.path.exists(probs_path):
        print(f'WARNING: {model_name} predictions not found at {model_dir}')
        continue
    
    all_probs[model_name] = np.load(probs_path)
    all_preds[model_name] = np.load(preds_path)
    labels = np.load(labels_path)
    
    if all_labels is None:
        all_labels = labels
    
    # Load F1 for weighted averaging
    results_path = os.path.join(model_dir, 'results.json')
    if os.path.exists(results_path):
        with open(results_path, 'r') as f:
            res = json.load(f)
        f1 = res.get('test_report', {}).get('macro avg', {}).get('f1-score', 0.0)
        all_val_f1[model_name] = f1
    else:
        from sklearn.metrics import f1_score
        all_val_f1[model_name] = f1_score(all_labels, all_preds[model_name], average='macro')
    
    print(f'{model_name:<20s}: {len(all_preds[model_name]):,} предсказаний | val F1={all_val_f1[model_name]:.4f}')

if len(all_probs) < 2:
    raise RuntimeError('Нужно как минимум 2 обученных модели. Запустите ноутбуки 02-04.')

In [ ]:
# Индивидуальные метрики каждой модели
from src.evaluation import evaluate_predictions, compare_models

individual_results = []
for model_name in all_probs:
    m = evaluate_predictions(
        all_labels, all_preds[model_name], all_probs[model_name],
        model_name=model_name
    )
    individual_results.append(m)
    print(f'{model_name}: accuracy={m["accuracy"]:.4f} | f1_macro={m["f1_macro"]:.4f}')

df_individual = pd.DataFrame(individual_results).set_index('model')
print('\n', df_individual[['accuracy', 'f1_macro', 'f1_weighted']].round(4))

## 2. Ансамблирование

In [ ]:
from src.ensemble import (
    hard_voting, soft_voting, weighted_averaging,
    stacking_ensemble, evaluate_ensemble,
)

probs_list = list(all_probs.values())
preds_list = list(all_preds.values())
f1_weights = list(all_val_f1.values())

print('Запускаю методы ансамблирования...')

In [ ]:
# ── Hard Voting ──────────────────────────────
print('\n[1] Hard Voting')
hv_preds = hard_voting(preds_list)
from sklearn.metrics import f1_score, accuracy_score
print(f'  Accuracy: {accuracy_score(all_labels, hv_preds):.4f}')
print(f'  F1-macro: {f1_score(all_labels, hv_preds, average="macro"):.4f}')

In [ ]:
# ── Soft Voting (equal weights) ───────────────
print('\n[2] Soft Voting')
sv_preds = soft_voting(probs_list)
print(f'  Accuracy: {accuracy_score(all_labels, sv_preds):.4f}')
print(f'  F1-macro: {f1_score(all_labels, sv_preds, average="macro"):.4f}')

In [ ]:
# ── Weighted Averaging (by F1) ────────────────
print('\n[3] Weighted Averaging (по F1-macro)')
print(f'  Веса: {dict(zip(all_probs.keys(), [f"{w:.3f}" for w in f1_weights]))}')
wa_preds = weighted_averaging(probs_list, f1_weights)
print(f'  Accuracy: {accuracy_score(all_labels, wa_preds):.4f}')
print(f'  F1-macro: {f1_score(all_labels, wa_preds, average="macro"):.4f}')

In [ ]:
# ── Stacking ─────────────────────────────────
# Для стекинга нужны validation предсказания как train для метамодели
# Загружаем val-предсказания
val_probs_list = []
val_labels = None

for model_name, model_dir in MODEL_DIRS.items():
    if model_name not in all_probs:
        continue
    # Если нет отдельных val_probs — используем часть test (первые 50%)
    # В production лучше сохранять val_probs отдельно в trainer.py
    val_probs_path = os.path.join(model_dir, 'val_probs.npy')
    if os.path.exists(val_probs_path):
        val_probs_list.append(np.load(val_probs_path))
        if val_labels is None:
            val_labels = np.load(os.path.join(model_dir, 'val_labels.npy'))
    else:
        # Fallback: используем первую половину test-набора как train для метамодели
        n = len(all_probs[model_name])
        val_probs_list.append(all_probs[model_name][:n//2])
        if val_labels is None:
            val_labels = all_labels[:n//2]

test_probs_half = [p[len(p)//2:] for p in probs_list]
test_labels_half = all_labels[len(all_labels)//2:]

print('\n[4] Stacking — Logistic Regression')
stack_lr_preds, _ = stacking_ensemble(
    val_probs_list, val_labels, test_probs_half, meta_learner='logistic'
)
print(f'  Accuracy: {accuracy_score(test_labels_half, stack_lr_preds):.4f}')
print(f'  F1-macro: {f1_score(test_labels_half, stack_lr_preds, average="macro"):.4f}')

In [ ]:
print('\n[5] Stacking — SVM')
stack_svm_preds, _ = stacking_ensemble(
    val_probs_list, val_labels, test_probs_half, meta_learner='svm'
)
print(f'  Accuracy: {accuracy_score(test_labels_half, stack_svm_preds):.4f}')
print(f'  F1-macro: {f1_score(test_labels_half, stack_svm_preds, average="macro"):.4f}')

## 3. Сравнение всех методов

In [ ]:
# Собираем результаты всех методов
ensemble_dict = {
    'Hard Voting':        hv_preds,
    'Soft Voting':        sv_preds,
    'Weighted Averaging': wa_preds,
}
df_ensemble = evaluate_ensemble(all_labels, ensemble_dict)

# Добавляем стекинг (на половине теста)
stacking_dict = {
    'Stacking (LogReg)': stack_lr_preds,
    'Stacking (SVM)':    stack_svm_preds,
}
df_stacking = evaluate_ensemble(test_labels_half, stacking_dict)

print('=== Основные ансамбли ===')
print(df_ensemble.round(4).to_string())
print('\n=== Стекинг ===')
print(df_stacking.round(4).to_string())

In [ ]:
# Финальная таблица: базовые модели + лучший ансамбль
all_results_rows = individual_results.copy()

best_ensemble_name = df_ensemble['f1_macro'].idxmax()
best_preds = ensemble_dict[best_ensemble_name]
best_metrics = evaluate_predictions(all_labels, best_preds, model_name=f'Ensemble ({best_ensemble_name})')
all_results_rows.append(best_metrics)

fig = compare_models(all_results_rows, save_path=f'{WORKING_DIR}/model_comparison.png')
print(f'\nЛучший ансамбль: {best_ensemble_name}')

In [ ]:
# Матрица ошибок лучшего ансамбля
from src.evaluation import plot_confusion_matrix
plot_confusion_matrix(
    all_labels, best_preds,
    model_name=f'Best Ensemble: {best_ensemble_name}',
    save_path=f'{WORKING_DIR}/cm_best_ensemble.png',
)

## 4. Итоги

In [ ]:
from sklearn.metrics import classification_report

print('\n' + '='*65)
print('ИТОГОВЫЙ ОТЧЁТ')
print('='*65)

LABEL_NAMES = ['negative', 'neutral', 'positive']

print('\n── Индивидуальные модели ──')
print(df_individual[['accuracy', 'f1_macro', 'f1_weighted']].round(4).to_string())

print('\n── Ансамбли ──')
print(df_ensemble.round(4).to_string())

print(f'\n── Лучший результат: {best_ensemble_name} ──')
print(classification_report(all_labels, best_preds, target_names=LABEL_NAMES))

print(f'\nВсе результаты и графики сохранены в: {WORKING_DIR}')

In [ ]:
# Сохраняем финальный summary
summary = {
    'individual_models': df_individual.round(4).to_dict(),
    'ensemble_methods':  df_ensemble.round(4).to_dict(),
    'best_ensemble':     best_ensemble_name,
    'best_f1_macro':     float(best_metrics['f1_macro']),
}
import json
with open(f'{WORKING_DIR}/final_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Summary saved.')